# LIFE on Google Colab — PolitiFact++ (combined binary fake/real, LLaMA2-7B)

Binary fake/real over **all four** PolitiFact++ files, pooled by veracity: **fake = HF + MF**, **real = HR + MR**, with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

> Note: this **extends** the paper's protocol, which detects over the **LLM pair only** (MF=fake, MR=real). Pooling in the human-written sources (HF/HR) makes this an "all-PolitiFact++, binary by veracity" task — results aren't directly comparable to the paper's 0.900/0.882.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed. (The official `meta-llama/Llama-2-7b-hf` is gated and requires an approved access request + token.)

Scope: **PolitiFact++** only (**520** articles: 194 fake = HF 97 + MF 97; 326 real = HR 194 + MR 132). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'

# Combined binary task: pool ALL four files by veracity -
# fake = HF + MF (194), real = HR + MR (326), total 520. Reconstructed with LLaMA2-7B.
# Distinct *_combined paths so the MF-vs-MR run's artifacts on Drive are left intact.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_combined'   # HF_fake/MF_fake (fake) + HR_true/MR_true (real)
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_combined.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_combined.pt'  # NEW path -> Step 1 trains a fresh extractor
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_combined'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_combined.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_combined.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

In [ ]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [ ]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [ ]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
from huggingface_hub import login
login()

## Step 0 — Convert PolitiFact++ to the combined binary JSONL
`--subset combined` emits all four files pooled by veracity: `HF_fake.jsonl` (97) + `MF_fake.jsonl` (97) labelled **fake**, and `HR_true.jsonl` (194) + `MR_true.jsonl` (132) labelled **real** — 520 articles. Because MF/MR reuse the HF/HR source ids (and MR has in-file dup ids), the converter rewrites each record's `id` to a synthetic unique value (`HF_0`, `MF_0`, …) so the Step 2 `(id, label)` join can't collide.

In [ ]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_BIN}" --subset combined

## Step 1 — Key-sentence extraction (top-10)
Trains a **fresh** BERT fake/real classifier on the combined data (saved to the new `BERT_CKPT`), then keeps the **top-10** most impactful sentences per article (paper's k for PolitiFact++). This is the slowest step (a forward pass per sentence per article). `BERT_CKPT` is a new path, so Step 1 retrains rather than loading the MF-vs-MR extractor (it skips training only if the file already exists).

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [ ]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

## Step 4 — Train the classifier (binary)
Splits `FEATURES_LLAMA` into train/test (random 80/20 → ~104 test items) and trains the Transformer+CRF classifier for **50 epochs** on the combined fake/real task. Reference points: the MF-vs-MR run reached **Acc 85.5 / Macro-F1 80.8**; the paper's MF-vs-MR target is 0.900 / 0.882 (not directly comparable to this pooled setup).

In [ ]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 50

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.